# Get Q2 2025's channel report from Gemini

*Data Analysis in Python · Topic 14.1*

Your manager wants Q2 2025's sessions, transactions, revenue and conversion rate by channel group, and you'll let Gemini write the code. Describe the table to it first, with the definitions it needs.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell and the two cells that build the `sessions` table.
2. In the first cell marked ✏️, write `describe_table(df)`: a function that returns a description of any DataFrame as text. Its first line is the number of rows and columns. Then one line per column, with the column's name and its type, and:
   - for numbers and dates, the smallest and the largest value
   - for anything else, the number of different values and the three most common ones

   Then create `context`: the description of the `sessions` table.
3. In the second cell marked ✏️, create `prompt`: a text with what the table is, `context`, your definitions of conversion rate and revenue, and the question. Ask for a table called `channels`, with one row per channel group, the columns `channel_group`, `sessions`, `transactions`, `revenue` and `conversion_rate`, sorted by revenue, highest first. Print it.
4. Open the Gemini panel, paste your prompt, and insert the code it writes into the third cell marked ✏️. Read the code, and run it. If it isn't right, tell Gemini what's wrong, or fix it yourself.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `describe_table(sessions)` starts with `5,315 rows and 9 columns:`. The `channels` table has 9 channel groups, with 287,371 sessions, 7,257 transactions and \$560,188.17 of revenue, sorted by revenue, from Organic Search (\$141,926.93) to Display. Email converts best, at 5.90%.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2025-04", "2025-05", "2025-06"]
START, END = "2025-04-01", "2025-06-30"  # Q2 2025

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}
raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Channel groups, with topic 4.2's steps and Foundations topic 12.3's function. Run it, and leave it as it is.
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium, with Foundations topic 12.3's rules."""
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"


def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])


sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions["channel_group"].value_counts()

## Describe the table

In [ ]:
def describe_table(df):
    """Return a short description of df for an AI assistant: its size, and each column's type and values."""
    lines = [f"{len(df):,} rows and {len(df.columns)} columns:"]
    for column in df.columns:
        values = df[column]
        if pd.api.types.is_numeric_dtype(values) or pd.api.types.is_datetime64_any_dtype(values):
            lines.append(f"- {column} ({values.dtype}): from {values.min()} to {values.max()}")
        else:
            examples = ", ".join(values.value_counts().index[:3])
            lines.append(f"- {column} ({values.dtype}): {values.nunique()} different values, such as {examples}")
    return "\n".join(lines)


context = describe_table(sessions)
print(context)

## The prompt

In [ ]:
prompt = f"""I'm analyzing a pandas DataFrame called sessions in a Colab notebook.
It's a GA4 export with one row per day, source, medium and campaign:

{context}

Definitions:
- Conversion rate is total transactions divided by total sessions, never a mean of rates.
- Revenue includes refunds as negative rows. Keep them.

Write pandas code that creates a table called channels, with one row per channel group,
the columns channel_group, sessions, transactions, revenue and conversion_rate,
sorted by revenue, highest first."""
print(prompt)

## Gemini's code

In [ ]:
# The kind of code Gemini writes for this prompt. Yours may look different and still be right.
channels = sessions.groupby("channel_group", as_index=False)[["sessions", "transactions", "revenue"]].sum()
channels["conversion_rate"] = channels["transactions"] / channels["sessions"]
channels = channels.sort_values("revenue", ascending=False).reset_index(drop=True)
channels

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    elif checked_before:
        results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
    elif [source for source in history if source.strip() and history.count(source) > 1]:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        first_line = repeated[0].strip().splitlines()[0]
        results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
    elif error is not None:
        results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
    else:
        results.append((True, "Every cell in this session ran once, with no errors."))

    describe_table = found.get("describe_table")
    if not callable(describe_table):
        results.append((False, "describe_table should be a function: def describe_table(df): ..."))
    else:
        sample = pd.DataFrame({
            "day": pd.to_datetime(["2025-05-01", "2025-05-02", "2025-05-02"]),
            "medium": ["cpc", "email", "cpc"],
            "clicks": [5, 7, 9],
        })
        try:
            text = describe_table(sample)
        except Exception as problem:
            text = f"{type(problem).__name__}: {problem}"
        lines = text.splitlines() if isinstance(text, str) else []

        def line_for(column):
            return next((line for line in lines if column in line), "")

        if not isinstance(text, str):
            results.append((False, "describe_table() should return the description as text: return \"\\n\".join(lines)."))
        elif not lines or "3 rows" not in lines[0]:
            results.append((False, f"describe_table() on a 3-row table should start with its size, such as '3 rows and 3 columns:'. It starts with {lines[0] if lines else text!r}."))
        elif [column for column in sample.columns if str(sample[column].dtype) not in line_for(column)]:
            results.append((False, "Each column's line should name the column and its type, such as '- clicks (int64): ...'."))
        elif "5" not in line_for("clicks") or "9" not in line_for("clicks") or "2025-05-01" not in line_for("day") or "2025-05-02" not in line_for("day"):
            results.append((False, f"Numbers and dates should show their smallest and largest values. For a clicks column from 5 to 9, the line is {line_for('clicks')!r}."))
        elif "2" not in line_for("medium") or "cpc" not in line_for("medium") or "email" not in line_for("medium"):
            results.append((False, f"Text columns should show how many different values they have, and the most common ones. For a medium column with cpc twice and email once, the line is {line_for('medium')!r}."))
        else:
            results.append((True, "describe_table() describes any table: its size, and each column's type and values."))

    context, prompt = found.get("context"), found.get("prompt")
    if not isinstance(context, str) or not context.startswith("5,315 rows"):
        results.append((False, "context should be describe_table(sessions): text that starts with '5,315 rows'."))
    elif not isinstance(prompt, str) or context not in prompt:
        results.append((False, "prompt should be text with context in it: put {context} inside an f-string."))
    elif "channels" not in prompt:
        results.append((False, "Ask for the table by name in the prompt: a table called channels."))
    else:
        results.append((True, "Your prompt describes the table, and asks for the channels table."))

    channels = found.get("channels")
    columns = ["channel_group", "sessions", "transactions", "revenue", "conversion_rate"]
    if not isinstance(channels, pd.DataFrame) or [column for column in columns if column not in channels.columns]:
        results.append((False, f"channels should be a DataFrame with the columns {', '.join(columns)}. If Gemini made channel_group the index, add .reset_index()."))
    else:
        revenue = float(channels["revenue"].sum())
        pooled = channels["transactions"] / channels["sessions"]
        if len(channels) != 9 or int(channels["sessions"].sum()) != 287371:
            results.append((False, f"Q2 2025 has 9 channel groups and 287,371 sessions. The channels table has {len(channels)} rows and {int(channels['sessions'].sum()):,} sessions. Use the whole sessions table."))
        elif abs(revenue - 640317.47) < 0.05:
            results.append((False, "The revenue is $640,317.47: the code dropped the refund rows, which have negative revenue. Keep them: the store's revenue is net of refunds."))
        elif abs(revenue - 560188.17) > 0.05:
            results.append((False, f"Q2 2025's revenue is $560,188.17. The channels table has ${revenue:,.2f}."))
        elif ((channels["conversion_rate"] - pooled).abs() > 0.00001).any():
            results.append((False, "The conversion rates aren't each channel group's transactions divided by its sessions. That's a mean of rates, row by row or day by day. Tell Gemini the definition, or fix the code."))
        elif channels.iloc[0]["channel_group"] != "Organic Search":
            results.append((False, "Sort the channels table by revenue, highest first: Organic Search comes first."))
        else:
            results.append((True, "9 channel groups, with 287,371 sessions and $560,188.17 of revenue. Email converts best, at 5.90%."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
def describe_table(df):
    """Return a short description of df for an AI assistant: its size, and each column's type and values."""
    lines = [f"{len(df):,} rows and {len(df.columns)} columns:"]
    for column in df.columns:
        values = df[column]
        if pd.api.types.is_numeric_dtype(values) or pd.api.types.is_datetime64_any_dtype(values):
            lines.append(f"- {column} ({values.dtype}): from {values.min()} to {values.max()}")
        else:
            examples = ", ".join(values.value_counts().index[:3])
            lines.append(f"- {column} ({values.dtype}): {values.nunique()} different values, such as {examples}")
    return "\n".join(lines)


context = describe_table(sessions)
print(context)
```

```python
prompt = f"""I'm analyzing a pandas DataFrame called sessions in a Colab notebook.
It's a GA4 export with one row per day, source, medium and campaign:

{context}

Definitions:
- Conversion rate is total transactions divided by total sessions, never a mean of rates.
- Revenue includes refunds as negative rows. Keep them.

Write pandas code that creates a table called channels, with one row per channel group,
the columns channel_group, sessions, transactions, revenue and conversion_rate,
sorted by revenue, highest first."""
print(prompt)
```

```python
# The kind of code Gemini writes for this prompt. Yours may look different and still be right.
channels = sessions.groupby("channel_group", as_index=False)[["sessions", "transactions", "revenue"]].sum()
channels["conversion_rate"] = channels["transactions"] / channels["sessions"]
channels = channels.sort_values("revenue", ascending=False).reset_index(drop=True)
channels
```

</details>